# Random Forest Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Optional for Random Forest)

**Note:** Random Forest is scale-invariant, meaning it doesn't require feature scaling. However, we'll still scale for consistency with other models.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## Random Forest Regression Formula and Concepts

**Random Forest Concept:**

Random Forest is an ensemble learning method that combines multiple decision trees to make predictions. It uses a technique called "bagging" (Bootstrap Aggregating).

**Bagging Process:**

1. **Bootstrap Sampling**: Create multiple random subsets of the training data (with replacement)
2. **Tree Training**: Train a decision tree on each subset
3. **Random Feature Selection**: At each split, only consider a random subset of features
4. **Aggregation**: Average predictions from all trees (for regression)

**Prediction Formula:**

$$\hat{y} = \frac{1}{N} \sum_{i=1}^{N} T_i(x)$$

Where:
- **ŷ** = final prediction
- **N** = number of trees in the forest
- **Tᵢ(x)** = prediction from the i-th tree
- **x** = input features

**Key Hyperparameters:**
- **n_estimators**: Number of trees in the forest (default=100)
- **max_depth**: Maximum depth of each tree
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples in a leaf node
- **max_features**: Number of features to consider at each split
- **bootstrap**: Whether to use bootstrap sampling (default=True)

**Advantages:**
- **Reduces overfitting**: By averaging multiple trees
- **Handles non-linearity**: Can capture complex patterns
- **Feature importance**: Provides feature importance scores
- **Robust to outliers**: Less sensitive than single decision tree
- **No scaling required**: Works with raw features
- **Parallelizable**: Trees can be trained in parallel

**Disadvantages:**
- **Less interpretable**: Harder to understand than single tree
- **Computationally expensive**: Training many trees takes time
- **Memory intensive**: Stores multiple trees in memory
- **Slower prediction**: Needs to average predictions from all trees

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train Random Forest with Default Parameters

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)

rf.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = rf.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Model Information

In [ ]:
print("Number of Trees:", rf.n_estimators)
print("Number of Features:", rf.n_features_in_)

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "Random Forest Regression"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Hyperparameter Tuning: Number of Trees

Let's test different numbers of trees to see how it affects performance.

In [ ]:
n_estimators_list = [10, 50, 100, 200, 500]

results = []

for n in n_estimators_list:
    model = RandomForestRegressor(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'n_estimators': n,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Visualize Performance vs Number of Trees

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(results_df['n_estimators'], results_df['R²'], 'o-')
plt.xlabel('Number of Trees')
plt.ylabel('R² Score')
plt.title('Random Forest Performance vs Number of Trees')
plt.grid(True, alpha=0.3)
plt.show()

## Compare with Single Decision Tree

Let's compare Random Forest with a single Decision Tree to see the improvement.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Single Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
r2_dt = r2_score(y_test, y_pred_dt)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)

print("Comparison Results:")
print(f"Single Decision Tree: R² = {r2_dt:.4f}")
print(f"Random Forest (100 trees): R² = {r2_rf:.4f}")
print(f"Improvement: {(r2_rf - r2_dt):.4f}")

## Summary

Random Forest Regression provides:
- **Reduced overfitting**: By averaging multiple decision trees
- **Better performance**: Usually outperforms single decision trees
- **Feature importance**: Identifies most important features
- **Robustness**: Less sensitive to noise and outliers
- **No scaling required**: Works with raw features

**Key advantages over single Decision Tree:**
- More stable and accurate predictions
- Less prone to overfitting
- Better generalization to new data

**Best practices:**
- Use cross-validation for hyperparameter tuning
- Start with default n_estimators=100
- Monitor feature importance for insights
- Consider computational cost for very large datasets